# 2-clean&filter

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "ID_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)
df.shape

(1128128, 27)

In [2]:
# Ne garder que le code style NORMAL
df = df[df["Code_style"] == "NORMAL"]

# Exclure les prises de parole de "Mme la présidente" et "M. le président"
# Role_debat n'est pas bien identifié, utiliser Nom_orateur
df = df[~df["Nom_orateur"].str.strip().isin(["M. le président", "Mme la présidente"])]

# Garder une trace de la longueur des interventions brutes
df["len_dirtytext"] = df["Texte"].str.len()

# Stabiliser le ID_orateur pour etre au format AN (pour matcher données)
# marche car pandas propage les NaN quand bien reconnu comme objet
# donc l'importance de str au chargement (et de pas forcer en str après ?)
df["ID_orateur"] = "PA" + df["ID_orateur"]

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["Code_parole"] = df["Code_parole"].fillna("non_précisé")

df.shape

(683680, 28)

In [3]:
df["ID_orateur"].isna().sum()  # 0

44952

In [4]:
# aperçu des répartitions
df.groupby("Code_parole", dropna=False)["len_dirtytext"].describe()

# TODO: voir à la toute fin si besoin de modifier les codes pris en compte avis_ ?

,count,mean,std,min,25%,50%,75%,max
Code_parole,,,,,,,,
(null),1.0,186.000000,NaN,186.0,186.00,186.0,186.00,186.0
AVIS_COM_1_10,1.0,1278.000000,NaN,1278.0,1278.00,1278.0,1278.00,1278.0
AVIS_COM_1_20,45107.0,450.390582,482.536522,3.0,118.50,315.0,618.00,8179.0
AVIS_GVT_1_20,38454.0,486.647371,681.527170,4.0,17.00,240.0,685.00,11340.0
PAROLE_1_1,21.0,556.761905,1080.603346,30.0,101.00,241.0,529.00,5088.0
PAROLE_1_2,247778.0,974.519219,1294.330875,3.0,245.00,547.0,1207.00,68092.0
PRESIDE_DISCOURS_1_10,2.0,95.500000,92.630988,30.0,62.75,95.5,128.25,161.0
Raccroche_apres_inter,9.0,1638.666667,1450.025431,60.0,154.00,1340.0,2316.00,3939.0
non_précisé,352287.0,282.342874,625.719777,1.0,17.00,37.0,251.00,25609.0


In [5]:
# TODO: regrouper les interventions interrompues ?

**?????????aviser pour regrouper les interventions interrompues ?????????**

## Match députés

### Match infos générales (historique)

In [6]:
df_deputes = pd.read_csv("../data/raw/id-dep/deputes-historique(datan-datagouv).csv")
# suppression des colonnes non utiles qui introduisent soucis parsing
df_deputes = df_deputes.drop(columns=["mail", "twitter", "facebook", "website"])


In [7]:
print("shape avant fusion:", df.shape)

assert df_deputes["id"].is_unique, "ids du df_deputes non uniques !"

# Merge et virer la col id pour éviter doublon
df = df.merge(
    df_deputes,
    left_on="ID_orateur",
    right_on="id",
    how="left",
    suffixes=("", "_dep"),
    validate="many_to_one",  # check if merge keys are unique in right dataset
).drop(columns=["id"])  # supprimer la colonne id du df_deputes

print("shape après fusion:", df.shape)

shape avant fusion: (683680, 28)
shape après fusion: (683680, 50)


### Match temporel des affiliations

### Recodage des dénominations
(Mais attention : ici choix de recoder avec nom des partis, alors que les groupes parlementaires sont + larges que les partis et peuvent servir à acceuillir des NI d'étiquettes diverses comme le groupe ECO qui acceuillent les députés de l'Après, Génération.s et Ruffin)

In [8]:
# recodage des grandes dénominations des groupes
# (moins sensible aux évolutions marginales de dénomination)

df_affiliation = pd.read_csv(
    "../data/raw/id-dep/datan_affiliations.csv", encoding="latin1", sep=";"
)  # format dégueu

# Recoder les partis pour stabilité temporelle des noms
# TODO: trancher pour dernires regroupements (ou garder pour ultérieur sur les blocs électoraux)
# Vérifier
recodage = {
    "RE": "REN",
    "LAREM": "REN",
    "DEM": "MODEM",
    "SOC": "PS",
    "SOC-A": "PS",
    "NG": "PS",
    "LFI-NUPES": "LFI",
    "FI": "LFI",
    "UDI-AGIR": "UDI",
    "UDI-A-I": "UDI",
    "LC": "UDI",
    "UDI_I": "UDI",
    "UDI-I": "UDI",
    "ECOLO": "ECO",
    "GDR-NUPES": "PCF",
    "GDR": "PCF",
    "LT": "LIOT",
    # Garde pour trace mais pas nécessaire car pas de changement
    # "LIOT": "LIOT",
    # "LR": "LR",
    # "RN": "RN",
    # "MODEM": "MODEM",
    # "LFI": "LFI",
    # "HOR": "HOR",
}

df_affiliation["libelleAbrev"] = df_affiliation["libelleAbrev"].astype(str).str.strip()
df_affiliation["parti_recod"] = df_affiliation["libelleAbrev"].replace(recodage)


In [ ]:
# Plutôt qu'un merge foireux parti sur un lookup ligne‑à‑ligne
# (= pb des orateurs non députés qui étaient pas présents, etc.)
# Le fichier est suffisamment réduit pour que le surplus de calcul soit pas un pb


# préparation des dates
df["DateSeance_ts"] = pd.to_datetime(
    df["DateSeance"], format="%Y%m%d%H%M%S%f", errors="raise"
)
df_affiliation["dateDebut"] = pd.to_datetime(
    df_affiliation["dateDebut"], errors="raise"
)
df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")
# # aviser si jamais besoin traiter affiliations en cours
# df_affiliation["dateFin"] = df_affiliation["dateFin"].fillna(pd.Timestamp("2100-01-01"))

# indexer par mpId pour lookup rapide
aff_by_mp = {
    mp: g[["dateDebut", "dateFin", "parti_recod"]].to_dict("records")
    for mp, g in df_affiliation.groupby("mpId")
}


def get_parti_for_row(row):
    mp = row.get("ID_orateur")  # correspond au mpId
    # gérer le cas des orateurs non députés ou autre type intervention
    if pd.isna(mp) or mp not in aff_by_mp:
        return None
    # récupérer le ts de l'intervention
    ts = row.get("DateSeance_ts")
    if pd.isna(ts):
        return None
    # retourner l'affiliation qui colle à la date d'intervention
    for rec in aff_by_mp[mp]:
        if rec["dateDebut"] <= ts <= rec["dateFin"]:
            return rec["parti_recod"]
    return None


# appliquer
df["parti_affiliation"] = df.apply(get_parti_for_row, axis=1)
# nope ne pas fill (car aussi des interv non députés ou autre type code parole)
# df["parti_affiliation"] = df["parti_affiliation"].fillna("UNKNOWN") # TODO: NOPE !! (NA CASES quand pas ID_orateur))


# Pas parfait mais pour avoir une idée :
print(
    "affectés :",
    df["parti_affiliation"].notna().sum(),
    # Eux on sait pas (pas députés, autre code parole intervention, etc.)
    "| non affectés :",
    df["parti_affiliation"].isna().sum(),
)

# TODO: envisager de forcer le renvoi de la derniere affiliation connue de df_deputes ?
# Aviser pour des matchs plus précis (cas limites, etc.) sur la base des repérages de matthias.


affectés : 529995 | non affectés : 153685


In [10]:
# Recodage des RN de la XVe législature au bloc RN
# = initialement en NI car pas assez nombreux pour former un groupe

liste_NI_RN = [
    "PA720822",
    "PA720668",
    "PA720468",
    "PA720614",
    "PA719436",
    "PA720802",
    "PA719608",
    "PA720606",
    "PA606212",
    "PA720798",
]

df.loc[df["ID_orateur"].isin(liste_NI_RN), "parti_affiliation"] = "RN"


In [11]:
df["parti_affiliation"].value_counts()

parti_affiliation
LR        123720
REN       122538
LFI        74724
PS         41151
PCF        40135
MODEM      36759
RN         28977
UDI        17249
LIOT       16738
ECO        14027
NI          5673
HOR         5053
AGIR-E      2334
EDS          917
Name: count, dtype: int64

In [12]:
df["groupeAbrev"].value_counts()

groupeAbrev
EPR          70220
DR           65347
LFI-NFP      50463
LAREM        47459
DEM          47189
LR           45043
RE           44629
SOC          31755
NI           25228
ECOS         23834
RN           23592
GDR          23002
LIOT         19604
GDR-NUPES    16755
HOR          13846
LFI-NUPES     9374
FI            6175
UDI_I         5788
SOC-A         5659
LT            5533
AGIR-E        4274
LES-REP       3307
UDR           1562
UMP            207
UDI-AGIR       182
ECOLO          144
NG             138
MODEM           43
Name: count, dtype: int64

In [13]:
# TODO: vérifier les modifs RN :
# On passe de
# RN          21684
# à
# RN         28977

# Et les NI de
# NI          12966
# à
# NI          5673

# 7293

# TODO: possible de checker contre groupeAbrev

In [14]:
# Sélectionner les députés RN selon l'affiliation
df_rn = df[df["parti_affiliation"] == "RN"]

# Comparer la colonne 'groupeAbrev' pour ces députés
# utiliser ou non drop_duplicates pour éviter les doublons selon but
comparison = df_rn[
    ["ID_orateur", "Nom_orateur", "parti_affiliation", "groupeAbrev"]
]  # .drop_duplicates()

# Afficher les différentes valeurs de groupeAbrev pour les RN
print(comparison["groupeAbrev"].value_counts())

comparison

groupeAbrev
RN    23405
NI     5572
Name: count, dtype: int64


,ID_orateur,Nom_orateur,parti_affiliation,groupeAbrev
333,PA719608,Mme Emmanuelle Ménard,RN,NI
371,PA719608,Mme Emmanuelle Ménard,RN,NI
373,PA719608,Mme Emmanuelle Ménard,RN,NI
405,PA719608,Mme Emmanuelle Ménard,RN,NI
510,PA719608,Mme Emmanuelle Ménard,RN,NI
...,...,...,...,...
683608,PA795836,M. Frédéric Cabrolier,RN,RN
683617,PA795836,M. Frédéric Cabrolier (RN),RN,RN
683619,PA795836,M. Frédéric Cabrolier,RN,RN
683649,PA795836,M. Frédéric Cabrolier (RN),RN,RN


## Export

In [15]:
# Export du csv nettoyé
df.to_csv("../data/interim/data_cleaning.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

# import csv  # pour utiliser csv.QUOTE_ALL et résoudre le soucis d'écart.
# df.to_csv(
#     "../data/interim/data_cleaning.csv",
#     index=False,
#     quoting=csv.QUOTE_ALL,  # permet de résoudre le soucis
# )

In [16]:
# verif ecriture/lecture ok
print("df shape:", df.shape)

df_test = pd.read_csv("../data/interim/data_cleaning.csv", low_memory=False)

print("df_test shape (après export import): ", df_test.shape)

df shape: (683680, 52)
df_test shape (après export import):  (683680, 52)


In [17]:
# TODO: regrouper les interventions interrompues ?

## Exploration

In [18]:
# explorer les afficiliation manquantes pour identifier les cas limites
# Notamment regarder ceux qui on pas d'affiliation mais bien un groupeAbrev
# Et aviser si on veut forcer l'affiliation à la derniere connue dans df_deputes
# En réalité sans doute des membres du gouvernement, donc toujours le même souci
# de décision à prendre selon l'usage qu'on veut faire des données

In [19]:
df_missing_affil = df[df["groupeAbrev"].notna() & df["parti_affiliation"].isna()]

In [20]:
df_missing_affil.shape

(60357, 52)

In [21]:
df_missing_affil[["ID_orateur", "Nom_orateur", "groupeAbrev", "DateSeance_ts"]]

,ID_orateur,Nom_orateur,groupeAbrev,DateSeance_ts
334,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
336,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
339,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
345,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
347,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
...,...,...,...,...
683670,PA721498,M. Stanislas Guerini,RE,2023-02-27 16:00:00
683672,PA721498,M. Stanislas Guerini,RE,2023-02-27 16:00:00
683675,PA721498,M. Stanislas Guerini,RE,2023-02-27 16:00:00
683677,PA721498,M. Stanislas Guerini,RE,2023-02-27 16:00:00


In [22]:
df_missing_affil["Nom_orateur"].nunique()

140

In [23]:
df_missing_affil["Nom_orateur"].unique()

array(['M. Bruno Le Maire', 'M. Edouard Philippe', 'M. Olivier Dussopt',
       'M. Benjamin Griveaux', 'M. Stéphane Travert', 'Mme Brune Poirson',
       'M.\xa0Christophe Castaner, secrétaire d’État et M.\xa0Éric Alauzet',
       'M. Christophe Castaner', 'M. Édouard Philippe',
       'M. Jean-Yves Le Drian', 'Mme Élisabeth Borne',
       'Mme Annick Girardin', 'M. Mounir Mahjoubi', 'M. Gérald Darmanin',
       'Mme Agnès Pannier-Runacher', 'M. Marc Fesneau',
       'M.\xa0Bruno Le\xa0Maire, ministre et M.\xa0Roland Lescure, rapporteur',
       'Mme Brigitte Bourguignon', 'M. Clément Beaune',
       'Mme Sarah El Haïry', 'Mme Geneviève Darrieussecq',
       'M. Franck Riester', 'M. François de Rugy',
       'Mme Amélie de Montchalin', 'M. Olivier Véran', 'Mme Nadia Hai',
       'Mme Roselyne Bachelot', 'M. Laurent Pietraszewski',
       'Mme Christelle Dubos', 'M. Gabriel Attal', 'M. Adrien Taquet',
       'M. Jean-Baptiste Djebbari', 'M. Éric Poulliat',
       'Mme Barbara Pompili',

In [24]:
df_missing_affil["Nom_orateur"].value_counts()[:50]

Nom_orateur
M. Gérald Darmanin              8130
M. Olivier Dussopt              5305
M. Bruno Le Maire               4402
Mme Agnès Pannier-Runacher      3970
Mme Élisabeth Borne             3928
M. Gabriel Attal                3242
M. Olivier Véran                2678
M. Marc Fesneau                 2270
M. Roland Lescure               1926
M. Adrien Taquet                1510
M. Édouard Philippe             1301
M. Stéphane Travert             1212
M. Laurent Pietraszewski        1198
Mme Brune Poirson               1168
M. Christophe Castaner          1049
M. François de Rugy             1049
M. Edouard Philippe             1025
Mme Catherine Vautrin            980
Mme Brigitte Bourguignon         951
M. Jean-Yves Le Drian            882
M. Jean-Noël Barrot              814
M. Franck Riester                765
Mme Olivia Grégoire              709
Mme Christelle Dubos             697
M. Clément Beaune                693
Mme Dominique Faure              580
M. Thomas Cazenave        

In [25]:
# TODO: harmoniser les NOM ORATEURS (\xa0, noms de groupe entre parenthèses, etc.)
# TODO: faire un fuzzyfuzz moche au besoin ?